# CHIRPS Rainfall Analysis over Ghana (2010–2020)

**Data source:** Climate Hazards Group InfraRed Precipitation with Station data (CHIRPS)

This notebook analyzes daily CHIRPS precipitation over Ghana for 2010–2020. It covers monthly and seasonal rainfall climatology, regional rainfall patterns, SPI-12 drought/wetness conditions, rainfall trends using the Mann–Kendall test and Sen’s slope, and agro-climatological metrics including rainfall onset, cessation, and length of growing period (LGP).

> **Reproducibility:** The source CHIRPS NetCDF and Ghana administrative boundary files are not embedded in this notebook. Place them in the paths specified below before running the analysis.

## 1. Setup

Install the required packages with `pip install -r requirements.txt` before running the notebook.

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rioxarray  # noqa: F401
import seaborn as sns
import xarray as xr
from scipy.stats import gamma, norm
import pymannkendall as mk

# Project paths
DATA_DIR = Path('data')
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHIRPS_FILE = DATA_DIR / 'Ghana_chirps.nc'
GHANA_SHP = DATA_DIR / 'boundaries' / 'gadm41_GHA_1.shp'
START_DATE = '2010-01-01'
END_DATE = '2020-12-31'

plt.rcParams.update({'figure.dpi': 110})


## 2. Load and inspect CHIRPS data

In [ ]:
ds = xr.open_dataset(CHIRPS_FILE)
print(ds)

# Select the analysis period
rain = ds.sel(time=slice(START_DATE, END_DATE))
precip = rain['precip']

print(f'Analysis period: {rain.time.min().values} to {rain.time.max().values}')
print(f'Grid: {precip.sizes["latitude"]} × {precip.sizes["longitude"]} cells')
print(f'Precipitation units: {precip.attrs.get("units", "not specified")}')


## 3. Prepare Ghana boundary and spatial metadata

In [ ]:
regions = gpd.read_file(GHANA_SHP)

if regions.crs is None:
    regions = regions.set_crs(epsg=4326)
elif regions.crs.to_epsg() != 4326:
    regions = regions.to_crs(epsg=4326)

region_col = 'NAME_1' if 'NAME_1' in regions.columns else regions.columns[1]

precip = precip.rio.write_crs('EPSG:4326')
precip = precip.rio.set_spatial_dims(x_dim='longitude', y_dim='latitude')

precip_ghana = precip.rio.clip(regions.geometry, regions.crs, drop=True)
print(f'Number of administrative regions: {len(regions)}')


## 4. Monthly rainfall climatology

Daily rainfall is aggregated to monthly totals. The climatological value for each calendar month is then calculated as the mean across 2010–2020.

In [ ]:
monthly_precip = precip_ghana.resample(time='MS').sum()
monthly_climatology = monthly_precip.groupby('time.month').mean(dim='time')

monthly_climatology = monthly_climatology.rio.write_crs('EPSG:4326')
monthly_climatology = monthly_climatology.rio.set_spatial_dims(
    x_dim='longitude', y_dim='latitude'
)

month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

fig, axes = plt.subplots(3, 4, figsize=(15, 11), sharex=True, sharey=True)
axes = axes.ravel()
vmin, vmax = 0, float(monthly_climatology.max())

for month in range(1, 13):
    ax = axes[month - 1]
    monthly_climatology.sel(month=month).plot(
        ax=ax, cmap='YlGnBu', vmin=vmin, vmax=vmax, add_colorbar=False
    )
    regions.boundary.plot(ax=ax, linewidth=0.5)
    ax.set_title(month_names[month - 1])
    ax.set_xlabel('')
    ax.set_ylabel('')

fig.colorbar(axes[0].collections[0], ax=axes, shrink=0.75, label='Mean monthly rainfall (mm)')
fig.suptitle('Ghana Monthly Rainfall Climatology (2010–2020)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'monthly_rainfall_climatology_2010_2020.png', dpi=300, bbox_inches='tight')
plt.show()


## 5. Seasonal rainfall climatology

In [ ]:
# Seasonal totals are computed from monthly rainfall totals.
seasonal_mean = monthly_precip.groupby('time.season').mean(dim='time') * 3
seasonal_mean = seasonal_mean.rio.write_crs('EPSG:4326')
seasonal_mean = seasonal_mean.rio.set_spatial_dims(x_dim='longitude', y_dim='latitude')

seasons = ['DJF', 'MAM', 'JJA', 'SON']
season_titles = {
    'DJF': 'DJF (Dec–Feb)',
    'MAM': 'MAM (Mar–May)',
    'JJA': 'JJA (Jun–Aug)',
    'SON': 'SON (Sep–Nov)'
}

fig, axes = plt.subplots(2, 2, figsize=(12, 10), sharex=True, sharey=True)
axes = axes.ravel()
vmin, vmax = 0, float(seasonal_mean.max())

for ax, season in zip(axes, seasons):
    im = seasonal_mean.sel(season=season).plot(
        ax=ax, cmap='YlGnBu', vmin=vmin, vmax=vmax, add_colorbar=False
    )
    regions.boundary.plot(ax=ax, linewidth=0.5)
    ax.set_title(season_titles[season], fontweight='bold')

fig.colorbar(im, ax=axes, shrink=0.75, label='Mean seasonal rainfall (mm)')
fig.suptitle('Ghana Seasonal Rainfall Climatology (2010–2020)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'seasonal_rainfall_climatology_2010_2020.png', dpi=300, bbox_inches='tight')
plt.show()


## 6. Regional monthly rainfall climatology

In [ ]:
regional_monthly = {}

for _, row in regions.iterrows():
    name = row[region_col]
    try:
        clipped = monthly_climatology.rio.clip([row.geometry], regions.crs, drop=True)
        regional_monthly[name] = clipped.mean(dim=['latitude', 'longitude']).values
    except Exception as exc:
        print(f'Skipping {name}: {exc}')

df_regional = pd.DataFrame(regional_monthly, index=month_names).T
df_regional.index.name = 'Region'
df_regional.to_csv(OUTPUT_DIR / 'ghana_regional_monthly_rainfall_2010_2020.csv')

plt.figure(figsize=(12, 8))
sns.heatmap(
    df_regional, cmap='YlGnBu', annot=True, fmt='.1f',
    cbar_kws={'label': 'Mean monthly rainfall (mm)'}, linewidths=0.5
)
plt.title('Ghana Regional Monthly Rainfall Climatology (2010–2020)', fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Region')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'regional_monthly_rainfall_heatmap_2010_2020.png', dpi=300, bbox_inches='tight')
plt.show()


## 7. Regional mean annual rainfall

In [ ]:
annual_precip = monthly_precip.groupby('time.year').sum(dim='time').mean(dim='year')
annual_precip = annual_precip.rio.write_crs('EPSG:4326')
annual_precip = annual_precip.rio.set_spatial_dims(x_dim='longitude', y_dim='latitude')

region_means = []
for _, row in regions.iterrows():
    try:
        clipped = annual_precip.rio.clip([row.geometry], regions.crs, drop=True)
        region_means.append(float(clipped.mean()))
    except Exception as exc:
        print(f'Could not calculate {row[region_col]}: {exc}')
        region_means.append(np.nan)

regions_annual = regions.copy()
regions_annual['annual_rainfall_mm'] = region_means
regions_annual.to_file(OUTPUT_DIR / 'ghana_regions_annual_rainfall.gpkg', driver='GPKG')

fig, ax = plt.subplots(figsize=(8, 10))
regions_annual.plot(
    column='annual_rainfall_mm', ax=ax, cmap='viridis', legend=True,
    legend_kwds={'label': 'Mean annual rainfall (mm)'}, edgecolor='black', linewidth=0.5
)
for _, row in regions_annual.iterrows():
    point = row.geometry.representative_point()
    ax.annotate(row[region_col], (point.x, point.y), ha='center', fontsize=6)
ax.set_title('Ghana Mean Annual Rainfall by Region (2010–2020)', fontweight='bold')
ax.set_axis_off()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'regional_annual_rainfall_2010_2020.png', dpi=300, bbox_inches='tight')
plt.show()


## 8. SPI-12 drought and wetness analysis

SPI-12 is calculated from the Ghana-average monthly precipitation series. A gamma distribution is fitted separately for each calendar month, accounting for zero-rainfall probability.

In [ ]:
monthly_series = (
    precip_ghana.resample(time='MS').sum()
    .mean(dim=['latitude', 'longitude'])
    .to_series()
)

def calculate_spi(series, scale=12):
    rolled = series.rolling(window=scale, min_periods=scale).sum().dropna()
    spi = pd.Series(index=rolled.index, dtype=float)

    for month in range(1, 13):
        mask = rolled.index.month == month
        sub = rolled.loc[mask]
        if sub.empty:
            continue

        zero_probability = (sub == 0).mean()
        positive = sub[sub > 0]
        if positive.empty:
            continue

        shape, _, scale_param = gamma.fit(positive, floc=0)
        cdf = gamma.cdf(sub, shape, loc=0, scale=scale_param)
        cdf = zero_probability + (1 - zero_probability) * cdf
        spi.loc[mask] = norm.ppf(np.clip(cdf, 1e-6, 1 - 1e-6))

    return spi

spi_12 = calculate_spi(monthly_series, scale=12)
df_spi = pd.DataFrame({'SPI_12': spi_12})
df_spi.to_csv(OUTPUT_DIR / 'ghana_spi12_2010_2020.csv')

fig, ax = plt.subplots(figsize=(12, 5))
wet = spi_12 >= 0
dry = spi_12 < 0
ax.bar(spi_12.index[wet], spi_12[wet], width=20, label='Wet')
ax.bar(spi_12.index[dry], spi_12[dry], width=20, label='Dry')
ax.axhline(0, linewidth=0.8)
ax.axhline(1.5, linestyle='--', linewidth=0.8, label='Very wet (+1.5)')
ax.axhline(-1.5, linestyle='--', linewidth=0.8, label='Severe drought (-1.5)')
ax.set_title('12-Month Standardized Precipitation Index (SPI-12), Ghana (2010–2020)', fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('SPI')
ax.grid(True, linestyle=':', alpha=0.5)
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'spi12_2010_2020.png', dpi=300, bbox_inches='tight')
plt.show()


## 9. Annual rainfall trend: Mann–Kendall and Sen’s slope

In [ ]:
annual_grids = precip_ghana.resample(time='YS').sum()

lats = annual_grids.latitude.values
lons = annual_grids.longitude.values
sens_slope = np.full((len(lats), len(lons)), np.nan)
p_values = np.full((len(lats), len(lons)), np.nan)

data = annual_grids.values

for i in range(len(lats)):
    for j in range(len(lons)):
        ts = data[:, i, j]
        if np.isnan(ts).any() or np.var(ts) == 0:
            continue
        result = mk.original_test(ts)
        sens_slope[i, j] = result.slope
        p_values[i, j] = result.p

slope_da = xr.DataArray(
    sens_slope, coords={'latitude': lats, 'longitude': lons}, dims=['latitude', 'longitude']
).rio.write_crs('EPSG:4326')
slope_da = slope_da.rio.set_spatial_dims(x_dim='longitude', y_dim='latitude')
p_val_da = xr.DataArray(
    p_values, coords={'latitude': lats, 'longitude': lons}, dims=['latitude', 'longitude']
)

fig, ax = plt.subplots(figsize=(8, 9))
slope_da.plot(ax=ax, cmap='RdYlBu', center=0, cbar_kwargs={'label': 'Sen’s slope (mm/year)'})
regions.boundary.plot(ax=ax, linewidth=0.6)

sig = p_val_da < 0.05
lon_grid, lat_grid = np.meshgrid(lons, lats)
ax.scatter(lon_grid[sig.values], lat_grid[sig.values], s=6, marker='o', label='p < 0.05')
ax.set_title('Ghana Annual Rainfall Trend (2010–2020)', fontweight='bold')
ax.set_xlabel('Longitude (°E)')
ax.set_ylabel('Latitude (°N)')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'rainfall_trend_mann_kendall_2010_2020.png', dpi=300, bbox_inches='tight')
plt.show()


## 10. Rainfall onset, cessation and length of growing period

In [ ]:
daily_series = precip_ghana.mean(dim=['latitude', 'longitude']).to_series()

def calculate_agro_metrics(series):
    results = []

    for year, yr_data in series.groupby(series.index.year):
        onset_doy = np.nan
        for i in range(len(yr_data) - 2):
            doy = yr_data.index[i].dayofyear
            if doy >= 60 and yr_data.iloc[i:i+3].sum() >= 20:
                onset_doy = doy
                break

        cessation_doy = np.nan
        for i in range(len(yr_data) - 1, 2, -1):
            doy = yr_data.index[i].dayofyear
            if 240 <= doy <= 334 and yr_data.iloc[i-2:i+1].sum() >= 10:
                cessation_doy = doy
                break

        lgp = (cessation_doy - onset_doy) if pd.notna(onset_doy) and pd.notna(cessation_doy) else np.nan
        results.append({
            'Year': year,
            'Onset_DOY': onset_doy,
            'Cessation_DOY': cessation_doy,
            'LGP_Days': lgp
        })

    return pd.DataFrame(results).set_index('Year')

df_agro = calculate_agro_metrics(daily_series)
df_agro.to_csv(OUTPUT_DIR / 'ghana_agro_climatology_2010_2020.csv')

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].plot(df_agro.index, df_agro['Onset_DOY'], marker='o', label='Onset')
axes[0].plot(df_agro.index, df_agro['Cessation_DOY'], marker='s', label='Cessation')
axes[0].set_ylabel('Day of year')
axes[0].set_title('Rainfall Onset and Cessation (2010–2020)', fontweight='bold')
axes[0].grid(True, linestyle=':', alpha=0.5)
axes[0].legend()

axes[1].bar(df_agro.index, df_agro['LGP_Days'], width=0.5)
mean_lgp = df_agro['LGP_Days'].mean()
axes[1].axhline(mean_lgp, linestyle='--', label=f'Mean LGP ({mean_lgp:.0f} days)')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('LGP (days)')
axes[1].set_title('Length of Growing Period', fontweight='bold')
axes[1].grid(True, linestyle=':', alpha=0.5)
axes[1].legend()

plt.suptitle('Agro-Climatological Metrics from CHIRPS Rainfall', fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'agro_climatological_metrics_2010_2020.png', dpi=300, bbox_inches='tight')
plt.show()


## 11. Key outputs

The analysis produces:

- Monthly rainfall climatology maps
- Seasonal rainfall climatology maps
- Regional monthly rainfall heatmap and CSV
- Regional mean annual rainfall map
- SPI-12 time series and CSV
- Pixel-wise Mann–Kendall p-values and Sen’s slope visualization
- Annual rainfall onset, cessation and length-of-growing-period CSV and figure

### Important methodological note
The onset and cessation definitions used here are threshold-based: onset is the first 3-day period with at least 20 mm after day 60, while cessation is identified backward using a 3-day total of at least 10 mm between days 240 and 334. These thresholds should be reported explicitly and, if this becomes a research publication, justified against established agro-climatological methods for the study region.